# StressSense — Workstream A on Colab (no local dataset)

Runs the **A pipeline** (WESAD -> windows -> LOSO/random scores) on Colab's **ephemeral** disk.
The ~2.25 GB dataset lives only in this runtime and is wiped when it ends — nothing is stored on your machine.
You download only the small `scores.csv` / `windows.parquet` at the end.

Run the cells top to bottom. Use a **GPU/CPU** runtime; no GPU needed.


## 1. Clone the repo and install dependencies


In [ ]:
!git clone https://github.com/Avijeettelkar1/stresssense.git
%cd stresssense
!git checkout a/data-model
!pip -q install numpy pandas scikit-learn pyarrow pyyaml neurokit2

## 2. Download WESAD to the ephemeral runtime (~2.25 GB)

Downloads to `/content` (Colab temp disk), not your machine. `wget -c` resumes if interrupted.
If the primary link fails, uncomment the fallback.


In [ ]:
# Primary link (used by several WESAD reproducibility repos):
!wget -c -O /content/WESAD.zip "https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download"
# Fallback (uncomment if the primary is down):
# !wget -c -O /content/WESAD.zip "https://uni-siegen.sciebo.de/s/pYjSgfOVs6Ntahr/download"
!unzip -q -o /content/WESAD.zip -d /content/
!ls /content/WESAD

## 3. Point the pipeline at the data

Reuses the committed `config.example.yaml` and only swaps the data path, so all your
windowing / model / label settings come along.


In [ ]:
cfg = open('config.example.yaml').read().replace('CHANGE_ME/path/to/WESAD', '/content/WESAD')
open('config.local.yaml', 'w').write(cfg)
print(cfg)

## 4. Verify the pickle layout matches the loader

Prints one subject's keys/shapes/label codes. If anything differs from `load_wesad.py`,
fix it there before the full run (this is your friend's open question, answered with data).


In [ ]:
!python scripts/inspect_wesad.py S2

## 5. Quick test on two subjects


In [ ]:
!python scripts/run_pipeline.py --stage data --subjects S2 S3

## 6. Full run: windows -> scores -> metrics (RQ1 / RQ2)

Takes a few minutes (physio features process every window). To go faster, set
`features: {physio: false}` in the config cell above.


In [ ]:
!python scripts/run_pipeline.py --stage all

## 7. Take the small hand-off files with you

`scores.csv` is the A->B contract; `windows.parquet` is A->B too. Both are < 1 MB.
Option A downloads them to your computer; Option B saves to Google Drive.


In [ ]:
# Option A: download to your computer
from google.colab import files
files.download('artifacts/scores.csv')
files.download('artifacts/windows.parquet')

In [ ]:
# Option B: save to Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# !cp artifacts/scores.csv artifacts/windows.parquet /content/drive/MyDrive/

---
**Hand-off:** drop `scores.csv` into your repo's `artifacts/` (git-ignored) locally, and B/C can
run against the real scores instead of the mocks. Or share it so B can start on real data.
